In [ ]:
# Cell 1: Install the PEFT Stack
!pip install -q -U transformers
!pip install -q -U peft
!pip install -q -U bitsandbytes
!pip install -q -U trl
!pip install -q -U datasets
!pip install -q -U accelerate

print("✅ PEFT Stack successfully installed!")

In [ ]:
# Cell 2: Authenticate and Load Data
from huggingface_hub import login
from datasets import load_dataset
import os

# Put your Hugging Face Write Token here
hf_token = "Your_Hugging_Face_Write_Token"
login(token=hf_token)

# Replace with the actual Repo ID you used in Week 8
# e.g., "your-username/pii-redactor-training-v1"
REPO_ID = "Your_Username/pii-redactor-training-v1"

print(f"Pulling {REPO_ID} from the Hub...")
# Load the dataset directly into the Colab GPU memory
dataset = load_dataset(REPO_ID, split="train")

print(f"✅ Successfully loaded {len(dataset)} training rows.")
print("\nSample Row:")
print(dataset[0]['messages'])

In [ ]:
# Cell 3: Load the Tokenizer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# We will use Mistral 7B Instruct v0.3
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_auth_token=hf_token)

# Setting up padding (Critical for batch training)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fixes weird formatting bugs during training

print("✅ Tokenizer loaded successfully!")

In [ ]:
# Cell 4: Define the 4-bit Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True, # Squeezes out even more memory!
    bnb_4bit_quant_type="nf4",      # NormalFloat4 - optimized for neural net weights
    bnb_4bit_compute_dtype=torch.bfloat16 # The math is still done in 16-bit for accuracy
)
print("✅ BitsAndBytes Configuration set!")

In [ ]:
# Cell 5: Load the Base Model
print("Downloading Base Model into 4-bit memory... (This takes a few minutes)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto", # Automatically puts the model on the GPU
    token=hf_token
)

# Disable caching to save VRAM during training
model.config.use_cache = False

print("✅ Model successfully loaded into 4-bit VRAM!")

In [ ]:
# Cell 6: Prepare for k-bit Training
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# Enable gradient checkpointing to save massive amounts of VRAM
model.gradient_checkpointing_enable()

# Prep the model for quantized training
model = prepare_model_for_kbit_training(model)
print("✅ Model prepped for gradient calculations.")

In [ ]:
# Cell 7: Define and Attach the LoRA Adapters
peft_config = LoraConfig(
    r=16,                       # The rank of the update matrices
    lora_alpha=32,              # The scaling factor (usually 2x the rank)
    target_modules=[            # Target all linear layers for maximum intelligence
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,          # Drop 5% of neurons randomly to prevent overfitting
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach the adapters to the 4-bit base model
model = get_peft_model(model, peft_config)

# Let's see exactly how many parameters we are actually training
model.print_trainable_parameters()

In [ ]:
# Cell 8: Dataset Formatting
def format_chat_template(row):
    # This uses the tokenizer to apply Llama-3's native <|start_header_id|> tags
    chat = row['messages']
    formatted_prompt = tokenizer.apply_chat_template(chat, tokenize=False)
    return {"text": formatted_prompt}

print("Formatting dataset to match Llama-3's prompt structure...")
formatted_dataset = dataset.map(format_chat_template)
print("✅ Formatting complete!")

In [ ]:
# Cell 9: Set up Training Arguments and Trainer
from trl import SFTConfig, SFTTrainer

training_arguments = SFTConfig(
    output_dir="./pii_redactor_checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=25,
    logging_steps=5,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    lr_scheduler_type="constant",
    train_sampling_strategy="group_by_length",
    dataset_text_field="text",
    max_length=1024
)

print("Initializing SFTTrainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
    args=training_arguments,
)
print("✅ Trainer ready.")

In [ ]:
# Cell 10: TRAIN
print("🚀 Commencing Neural Surgery (Training Started)...")
trainer.train()